# Integratie van deelvraag 3 en 4 – Voorspelling en implicaties

In dit notebook worden de resultaten van **deelvraag 3 (Bo)** en **deelvraag 4 (Noah)** samengebracht.  
Het doel is om de voorspellingen per sector logisch te combineren met de interpretatie van die voorspellingen, zodat er één geheel ontstaat dat aansluit op dezelfde werkwijze als in Faysal zijn integratienotebook.

Dit notebook vormt daarmee de brug tussen:
- de **modellering** van de werkgelegenheidsontwikkeling per sector;
- en de **inhoudelijke interpretatie** van de gevolgen voor de economische structuur van Flevoland.

## Inleiding

Binnen team 2 zijn twee onderdelen uitgewerkt:

- **Deelvraag 3 – Modellering:** welke methode kan worden gebruikt om de ontwikkeling per sector te voorspellen?
- **Deelvraag 4 – Interpretatie en implicaties:** wat betekenen deze voorspellingen voor Flevoland?

Het doel van deze integratie is vierledig:

1. de code en aanpak van Bo en Noah samenbrengen in één notebook;
2. controleren of de uitwerking logisch aansluit op de businessvraag;
3. de voorspellingen per sector overzichtelijk interpreteren;
4. een gezamenlijke conclusie formuleren voor de opdrachtgever.

De centrale businessvraag blijft daarbij:

> **Hoe zal de werkgelegenheid per sector in Flevoland zich naar verwachting ontwikkelen in de komende 10 jaar?**

## CRISP-DM plaatsing van dit notebook

Dit notebook valt vooral binnen twee fasen van CRISP-DM:

- **Modeling:** het opzetten van een voorspelmodel per sector;
- **Evaluation:** het interpreteren van de uitkomsten en het vertalen naar de businessvraag.

Daarom bevat dit notebook zowel de modelcode van Bo als de interpretatie- en sectoranalyse van Noah.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm

from pathlib import Path

# Prophet wordt gebruikt in Bo zijn notebook
from prophet import Prophet

## 1. Data inladen

Hieronder staat de basis van Bo en Noah hun notebooks: het inladen van de dataset.  
Pas het pad eventueel aan als de dataset lokaal ergens anders staat.

In [ ]:
# Pas dit pad aan als de dataset ergens anders staat
data_path = Path("../../Data/raw/dataset.xlsx")

df = pd.read_excel(data_path)
df.head()

## 2. Voorbereiding dataset voor voorspelling

Deze stap komt direct uit de notebook van Bo en vormt ook de basis voor Noah zijn interpretatie.  
We selecteren de relevante kolommen en vertalen de sectorcodes naar leesbare sectornamen.

In [ ]:
df_filtered = df[['vestnr', 'jaar', 'postcode', 'datum_start', 'datum_einde', 'wp', 'sector_code', 'sb']].copy()

sector_map = {
    'A': 'Landbouw, bosbouw en visserij',
    'B': 'Winning van delfstoffen',
    'C': 'Industrie',
    'D': 'Energievoorziening',
    'E': 'Water en afvalbeheer',
    'F': 'Bouwnijverheid',
    'G': 'Handel en reparatie van auto’s',
    'H': 'Vervoer en opslag',
    'I': 'Horeca',
    'J': 'Informatie en communicatie',
    'K': 'Financiële instellingen',
    'L': 'Verhuur en handel van onroerend goed',
    'M': 'Zakelijke dienstverlening',
    'N': 'Overige zakelijke dienstverlening',
    'O': 'Openbaar bestuur',
    'P': 'Onderwijs',
    'Q': 'Gezondheidszorg',
    'R': 'Cultuur, sport en recreatie',
    'S': 'Overige dienstverlening'
}

df_filtered['sector_code'] = df_filtered['sector_code'].map(sector_map)
df_filtered.head()

## 3. Historische aggregatie per sector en jaar

Net als in Bo en Noah hun notebooks berekenen we hier:

- het **aantal vestigingen/bedrijven** per sector per jaar;
- het totale aantal **werkzame personen (`wp`)** per sector per jaar.

Zo ontstaat één overzichtelijke tabel die gebruikt kan worden voor zowel modellering als interpretatie.

In [ ]:
df_aantal = (
    df_filtered
    .groupby(['sector_code', 'jaar'])
    .size()
    .reset_index(name='aantal')
)

df_wp = (
    df_filtered
    .groupby(['sector_code', 'jaar'])['wp']
    .sum()
    .reset_index(name='wp')
)

df_yearly = df_aantal.merge(df_wp, on=['sector_code', 'jaar'])

df_pivot = df_yearly.pivot(
    index='sector_code',
    columns='jaar',
    values=['aantal', 'wp']
)

df_pivot.columns = [f"{value}_{jaar}" for value, jaar in df_pivot.columns]

wp_cols = [col for col in df_pivot.columns if col.startswith('wp_')]
aantal_cols = [col for col in df_pivot.columns if col.startswith('aantal_')]

df_pivot[wp_cols] = df_pivot[wp_cols].fillna(0)
df_pivot[aantal_cols] = df_pivot[aantal_cols].fillna(0)

df_pivot['verschil_aantal'] = df_pivot['aantal_2024'] - df_pivot['aantal_2014']
df_pivot['pct_groei_aantal'] = np.where(
    df_pivot['aantal_2014'] != 0,
    (df_pivot['aantal_2024'] - df_pivot['aantal_2014']) / df_pivot['aantal_2014'] * 100,
    np.nan
)

df_pivot['verschil_wp'] = df_pivot['wp_2024'] - df_pivot['wp_2014']
df_pivot['pct_groei_wp'] = np.where(
    df_pivot['wp_2014'] != 0,
    (df_pivot['wp_2024'] - df_pivot['wp_2014']) / df_pivot['wp_2014'] * 100,
    np.nan
)

df_pivot = df_pivot.reset_index()
df_pivot.head()

## 4. Historische visualisaties

Deze visualisaties komen uit Bo zijn notebook en laten zien hoe sectoren zich historisch hebben ontwikkeld.

In [ ]:
df_pivot.set_index('sector_code')[aantal_cols].T.plot(
    kind='bar',
    stacked=True,
    figsize=(14, 8),
    colormap='tab20'
)
plt.title('Aantal vestigingen per sector per jaar (gestapeld)', fontsize=16)
plt.xlabel('Jaar', fontsize=12)
plt.ylabel('Aantal vestigingen', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Sector', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
df_pivot.set_index('sector_code')[wp_cols].T.plot(
    kind='bar',
    stacked=True,
    figsize=(14, 8),
    colormap='tab20'
)
plt.title('Werkzame personen (wp) per sector per jaar (gestapeld)', fontsize=16)
plt.xlabel('Jaar', fontsize=12)
plt.ylabel('Werkzame personen', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Sector', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
colors = cm.tab20(np.linspace(0, 1, len(df_pivot)))

plt.figure(figsize=(14, 6))
plt.bar(df_pivot['sector_code'], df_pivot['pct_groei_aantal'], color=colors)
plt.title('Percentage groei aantal vestigingen 2014 → 2024 per sector', fontsize=16)
plt.xlabel('Sector', fontsize=12)
plt.ylabel('Groei (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
plt.bar(df_pivot['sector_code'], df_pivot['pct_groei_wp'], color=colors)
plt.title('Percentage groei werkzame personen 2014 → 2024 per sector', fontsize=16)
plt.xlabel('Sector', fontsize=12)
plt.ylabel('Groei (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Modellering per sector (Bo)

Hieronder staat de kern van Bo zijn modelcode.  
Per sector wordt een tijdreeks opgebouwd en vervolgens met **Prophet** een voorspelling gemaakt voor de komende 10 jaar.

**Belangrijke methodologische opmerking:**  
In dit model wordt `y` gevuld met **aantal vestigingen** en wordt `wp` gebruikt als extra regressor.  
De voorspellingen gaan dus technisch gezien primair over het aantal vestigingen, terwijl `wp` wel wordt meegenomen als verklarende variabele.  
Voor de interpretatie houden we daar rekening mee.

In [ ]:
def prepare_prophet_df(df_pivot, sector):
    df_sector = df_pivot[df_pivot['sector_code'] == sector].copy()

    jaren = range(2014, 2025)
    data = {'ds': [], 'y': [], 'wp': []}

    for jaar in jaren:
        data['ds'].append(pd.to_datetime(str(jaar)))
        data['y'].append(df_sector[f'aantal_{jaar}'].values[0])
        data['wp'].append(df_sector[f'wp_{jaar}'].values[0])

    df_long = pd.DataFrame(data)
    return df_long


all_forecasts = []

for sector in df_pivot['sector_code']:
    df_prophet = prepare_prophet_df(df_pivot, sector)

    model = Prophet(yearly_seasonality=False)
    model.add_regressor('wp')
    model.fit(df_prophet[['ds', 'y', 'wp']])

    future = model.make_future_dataframe(periods=10, freq='YE')

    existing_wp = df_prophet['wp'].values
    wp_slope = (existing_wp[-1] - existing_wp[0]) / (2024 - 2014)
    future_wp = [existing_wp[-1] + wp_slope * (i + 1) for i in range(10)]

    future['wp'] = np.concatenate([existing_wp, future_wp])

    forecast = model.predict(future)
    forecast['sector_code'] = sector
    forecast['yhat'] = forecast['yhat'].apply(lambda x: max(round(x), 0))

    all_forecasts.append(forecast[['ds', 'yhat', 'sector_code']])

df_forecast_all = pd.concat(all_forecasts).reset_index(drop=True)
df_forecast_all.head()

## 6. Visualisatie van de voorspellingen per sector

Deze stap sluit aan op Bo zijn grafieken en vormt tegelijk de input voor Noah zijn interpretatie.

In [ ]:
sectoren = df_forecast_all['sector_code'].unique()
n_sectors = len(sectoren)
cols = 4
rows = (n_sectors + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
axes = axes.flatten()

for i, sector in enumerate(sectoren):
    df_plot = df_forecast_all[df_forecast_all['sector_code'] == sector]
    axes[i].plot(df_plot['ds'], df_plot['yhat'], marker='o')
    axes[i].set_title(f'Sector: {sector}')
    axes[i].set_xlabel('Jaar')
    axes[i].set_ylabel('Voorspeld aantal vestigingen')
    axes[i].grid(True, linestyle='--', alpha=0.5)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## 7. Vertaling van voorspellingen naar sectoranalyse (Noah)

In Noah zijn notebook worden de grafieken inhoudelijk geïnterpreteerd.  
Om daar een logischer en beter navolgbaar geheel van te maken, zetten we de voorspellingen nu ook om naar een overzichtstabel per sector.

Hiermee kunnen we sectoren indelen in:

- **groei**
- **stabiel**
- **krimp**

De indeling gebeurt hier op basis van de totale procentuele verandering tussen 2024 en 2034.

In [ ]:
df_forecast_all['jaar'] = pd.to_datetime(df_forecast_all['ds']).dt.year

df_future_summary = (
    df_forecast_all[df_forecast_all['jaar'].isin([2024, 2034])]
    .pivot(index='sector_code', columns='jaar', values='yhat')
    .reset_index()
    .rename(columns={2024: 'waarde_2024', 2034: 'waarde_2034'})
)

df_future_summary['absolute_verandering'] = (
    df_future_summary['waarde_2034'] - df_future_summary['waarde_2024']
)

df_future_summary['pct_verandering'] = np.where(
    df_future_summary['waarde_2024'] != 0,
    (df_future_summary['waarde_2034'] - df_future_summary['waarde_2024']) / df_future_summary['waarde_2024'] * 100,
    np.nan
)

def categoriseer_sector(pct):
    if pd.isna(pct):
        return 'onbekend'
    elif pct > 10:
        return 'groei'
    elif pct < -10:
        return 'krimp'
    else:
        return 'stabiel'

df_future_summary['categorie'] = df_future_summary['pct_verandering'].apply(categoriseer_sector)

df_future_summary = df_future_summary.sort_values('pct_verandering', ascending=False)
df_future_summary

In [ ]:
plt.figure(figsize=(14, 7))
sns.barplot(
    data=df_future_summary,
    x='pct_verandering',
    y='sector_code',
    hue='categorie'
)
plt.title('Voorspelde procentuele verandering per sector (2024 → 2034)', fontsize=16)
plt.xlabel('Verandering (%)', fontsize=12)
plt.ylabel('Sector', fontsize=12)
plt.axvline(10, linestyle='--')
plt.axvline(-10, linestyle='--')
plt.tight_layout()
plt.show()

## 8. Uitleg van de sectorcategorieën

De onderstaande code geeft dezelfde stap weer die Noah in tekst uitvoerde, maar nu in een reproduceerbare vorm.

In [ ]:
groeisectoren = df_future_summary[df_future_summary['categorie'] == 'groei']['sector_code'].tolist()
stabiele_sectoren = df_future_summary[df_future_summary['categorie'] == 'stabiel']['sector_code'].tolist()
krimpsectoren = df_future_summary[df_future_summary['categorie'] == 'krimp']['sector_code'].tolist()

print("Groeisectoren:")
for sector in groeisectoren:
    print("-", sector)

print("\nStabiele sectoren:")
for sector in stabiele_sectoren:
    print("-", sector)

print("\nKrimpsectoren:")
for sector in krimpsectoren:
    print("-", sector)

## 9. Controle op logica en consistentie

Als teamleider is het belangrijk om te controleren of de voorspellingen logisch aansluiten op de historische analyse.

Daarom vergelijken we:
- historische groei 2014–2024;
- voorspelde groei 2024–2034.

Zo zien we waar het model historische trends ongeveer doorzet en waar extra voorzichtigheid nodig is.

In [ ]:
df_logica = df_pivot[['sector_code', 'pct_groei_aantal', 'pct_groei_wp']].merge(
    df_future_summary[['sector_code', 'pct_verandering', 'categorie']],
    on='sector_code',
    how='left'
)

df_logica = df_logica.rename(columns={
    'pct_groei_aantal': 'historische_groei_aantal_2014_2024',
    'pct_groei_wp': 'historische_groei_wp_2014_2024',
    'pct_verandering': 'voorspelde_groei_2024_2034'
})

df_logica.sort_values('voorspelde_groei_2024_2034', ascending=False)

### Korte interpretatie van de controle

- Sectoren met **historische groei** én **voorspelde groei** zijn inhoudelijk het meest consistent.
- Sectoren met **historische daling** maar toch **voorspelde groei** vragen om extra controle.
- Sectoren met sterke schommelingen moeten voorzichtig worden geïnterpreteerd, omdat het model historische patronen doorzet maar externe factoren niet meeneemt.

## 10. Geïntegreerde interpretatie

Op basis van de code van Bo en de interpretatie van Noah ontstaat het volgende beeld:

### Sterk groeiende sectoren
Sectoren die zowel historisch als in de voorspelling relatief sterk naar voren komen, kunnen worden gezien als sectoren die waarschijnlijk belangrijker worden in de economie van Flevoland.

### Stabiele sectoren
Sectoren met beperkte verandering vormen een meer constante economische basis. Zij dragen minder bij aan extra groei, maar zorgen wel voor continuïteit.

### Krimpsectoren
Sectoren met een structurele daling of beperkte toekomstverwachting verliezen relatief aan belang binnen de economische structuur van Flevoland.

### Inhoudelijke betekenis
De algemene richting van de economie lijkt te verschuiven naar sectoren met meer dienstverlening, publieke functies en ondersteunende economische activiteiten. Traditionele of kleinere sectoren laten vaker stagnatie of krimp zien.

## 11. Methodologische aandachtspunten

Voor een eerlijke beoordeling van dit notebook horen ook de beperkingen erbij:

1. **Het model voorspelt primair het aantal vestigingen**, niet rechtstreeks de werkgelegenheid in `wp`.
2. `wp` wordt wel als regressor gebruikt, waardoor werkgelegenheid indirect invloed heeft op de voorspelling.
3. Externe factoren, zoals beleid, conjunctuur, inflatie of migratie, zijn niet meegenomen.
4. Er is nog geen aparte validatiestap opgenomen met bijvoorbeeld train-test-evaluatie of foutmaten.

Dat betekent dat de resultaten vooral **richtinggevend** zijn en niet als exacte uitkomst moeten worden gelezen.

## 12. Conclusie voor Flevoland

Door deelvraag 3 en deelvraag 4 samen te voegen ontstaat één logisch geheel:

- Bo levert de technische voorspelling per sector;
- Noah vertaalt die voorspellingen naar economische betekenis;
- deze integratie verbindt beide onderdelen met de businessvraag.

### Eindconclusie
De ontwikkeling in Flevoland wijst erop dat niet alle sectoren zich in hetzelfde tempo ontwikkelen. Een deel van de sectoren laat duidelijke groei zien, een deel blijft relatief stabiel en een kleiner deel laat krimp zien. Daardoor verandert de economische structuur van Flevoland geleidelijk.

Voor de opdrachtgever betekent dit dat beleid vooral relevant wordt op:
- het ondersteunen van groeisectoren;
- het voorbereiden van onderwijs en arbeidsmarkt op verschuivende vraag;
- en het monitoren van sectoren die onder druk staan.

Dit notebook geeft daarmee een geïntegreerde basis voor de beantwoording van de businessvraag.